In [1]:
import pandas as pd
import numpy as np
import random
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory
import yfinance as yf
from pathlib import Path
import os
import matplotlib.pyplot as plt


In [2]:
anos = ['2025-12-31',
 '2024-12-31',
 '2023-12-31',
 '2022-12-31',
 '2021-12-31',
 '2020-12-31',
 '2019-12-31',
 '2018-12-31',
 '2017-12-31',
 '2016-12-31',
 '2015-12-31',
 ]

In [3]:

# Retornos

dfs = {}
for ano in anos:
    ano_usado = ano.split('-')[0]
    ano_usado_mais_um = int(ano_usado)+1
    nome = f'df_ativos_{ano_usado}.csv'
    # caminho = Path('retornos_ativos_otimizacao') / nome
    dfs[ano_usado] = pd.read_csv('../dados_retornos/otimizacao/ativos/'+nome)
    dfs[ano_usado]['date'] = pd.to_datetime(dfs[ano_usado]['date'])
    dfs[ano_usado] = dfs[ano_usado].set_index('date')
    dfs[ano_usado] = dfs[ano_usado].drop(columns=dfs[ano_usado].columns[(dfs[ano_usado] == 0.0).all()])
    dfs[ano_usado] = dfs[ano_usado].loc[str(ano_usado_mais_um)]
    ativos_manter = (dfs[ano_usado]==0.0).mean()
    c_manter = ativos_manter[ativos_manter<0.5].index
    dfs[ano_usado] = dfs[ano_usado][c_manter]

    print(ano,'-',len(dfs[ano_usado].columns))

2025-12-31 - 100
2024-12-31 - 99
2023-12-31 - 99
2022-12-31 - 99
2021-12-31 - 97
2020-12-31 - 84
2019-12-31 - 77
2018-12-31 - 76
2017-12-31 - 73
2016-12-31 - 69
2015-12-31 - 68


In [4]:
vb_cardinalidade_max = 15
# vb_cardinalidade_min = 10
vb_peso_maximo = 0.15
# vb_peso_minimo = 0.02
vb_theta = 0.25

In [5]:
anos = ['2015-12-31', '2016-12-31', '2017-12-31', '2018-12-31', '2019-12-31', '2020-12-31', '2021-12-31', '2022-12-31', '2023-12-31', '2024-12-31', '2025-12-31']
print(anos)

['2015-12-31', '2016-12-31', '2017-12-31', '2018-12-31', '2019-12-31', '2020-12-31', '2021-12-31', '2022-12-31', '2023-12-31', '2024-12-31', '2025-12-31']


## -----------------------

In [6]:
y = []
carteiras_anuais = {}
melhor_pesos = None
historico = []
carteiras_criadas = []
for a in anos:
    y.append(a.split('-')[0])
print("=-"*48)
print("Anos totais: ",y)
print("=-"*48)
lista_ativos_finais = {}
for an in anos:
    ano = an.split("-")[0]
    ano_int = int(ano)	
    print("COMEÇANDO ANO NOVO: ",ano)
    #deletar modelo
    if 'model' in locals():
        del model
        print('Modelo Antigo deletado \n Iniciando Novo')
    else:
        print("Nao consta model")
        pass
    try:
        # retorno_usado = pd.DataFrame(eval(df_usado))
        df_usado = dfs[str(ano_int)]
        retorno_usado = df_usado.copy()

        score_todos_anos = pd.read_excel(f'../../../score-raspagem/piotroski/resultado/{ano_int}.xlsx')
        score_usado_p = score_todos_anos[['Unnamed: 0', 'p_norm']].set_index('Unnamed: 0').T
        
        ativos = retorno_usado.columns.intersection(score_usado_p.columns)
        retorno_usado = retorno_usado[ativos]
        retorno_usado = (retorno_usado-retorno_usado.min())/(retorno_usado.max()-retorno_usado.min())
        score = score_usado_p[ativos]

        # COV
        df_cov = retorno_usado.cov()

        # Lista ativos
        lista_ativos_finais[ano] = retorno_usado.columns.tolist()
      
    except Exception as e:
            print("=========ERRRRRRRRRRRROR")
            print(e)    

    print("# ------ CRIAÇÃO DO MODELO")
    print("## Carteira criada para o ano: ",ano_int+1)
    print("## UTILIZANDO DADOS DE RETORNO DE: ",ano)

    #  --------- MODELO
   

    #  --------- MODELO
    model = pyo.ConcreteModel()
    #---------VARIÁVEIS-----------
    model.nome_ativos = pyo.Set(initialize = retorno_usado.columns)
    model.ativos = pyo.RangeSet(0, len(retorno_usado.columns)-1)
    model.dias = pyo.RangeSet(0, len(retorno_usado)-1)
    model.retornos_ativos = pyo.Param(model.dias, model.ativos, initialize=lambda model,dia, ativo: retorno_usado.iloc[dia, ativo])    
    model.sigma = pyo.Param(model.ativos, model.ativos, initialize=lambda model,a,b: df_cov.iloc[a,b])
    # model.sigma_min = pyo.Param(initialize=0, mutable=True)
    # model.sigma_max = pyo.Param(initialize=1, mutable=True)
    model.score = pyo.Param(model.ativos, initialize=lambda model,a: score.iloc[0,a])
    model.theta = pyo.Param(initialize=0.5)

    # ------------------------------

    #linearizar Sharpe
    # z:= kx
    model.z = pyo.Var(model.ativos, domain=pyo.NonNegativeReals)
    model.k = pyo.Var(domain=pyo.NonNegativeReals)
    model.y = pyo.Var(model.ativos, domain=pyo.Binary)
    model.w = pyo.Var(model.ativos, domain=pyo.NonNegativeReals)
    model.KMAX = pyo.Param(initialize=100)

    # Com a problemática de cardinalidade y e bilinearidade, resolve-se
    # PESO
    # antes era model.x[a] <= model.pesomax * model.y[a]
    #  z=kx  -> model.z[a] <= model.pesomax * modely[a] * model.k[a]
    # CARDINALIDADE
    # era sum(model.y[a] ...) >= e <=
    # é feita a tranformação de w <= k * y

    # Com y=0 
    def res_y_zero(model,a):
            return model.w[a] <= model.KMAX * model.y[a]
    model.y_zero = pyo.Constraint(model.ativos, rule=res_y_zero)

    #--------------------------------------------------------------------------
    # com y=1
    #  segundo teto, quando y=1 ; w = k , pelas restriçoes >= e <=
    def res_y_1_teto2(model,a):
            return model.w[a] <= model.k 
    model.res_teto2 = pyo.Constraint(model.ativos, rule=res_y_1_teto2)

    # Limite inf 1
    def limite_1(model,a):
            return model.w[a] >= model.k - model.KMAX*(1-model.y[a])
    model.r_limite1 = pyo.Constraint(model.ativos, rule=limite_1)
    #--------------------------------------------------------------------------
    # #  PESOS MIN MAX
    # def pesomin(model,a):
    #         return model.z[a] >= 0.02 * model.w[a]
    # model.r_pesomin = pyo.Constraint(model.ativos, rule=pesomin)

    def pesomax(model,a):
            return model.z[a] <= 0.2 * model.w[a]
    model.r_pesomax = pyo.Constraint(model.ativos, rule=pesomax)

    # #  cardinalidade
    # def card_min(model):
    #         return sum(model.y[a] for a in model.ativos) >= vb_cardinalidade_min
    # model.r_cardmin = pyo.Constraint(rule=card_min)
    def card_max(model):
            return sum(model.y[a] for a in model.ativos) <= vb_cardinalidade_max
    model.r_card_max = pyo.Constraint(rule=card_max)

    # Restrições Sharpe
    def restricao_z(model):
            return sum(model.z[a] for a in model.ativos) == model.k
    model.r_restricao_z = pyo.Constraint(rule=restricao_z)

    def restricao_z_retorno(model):
            return sum(model.sigma[a,b]*model.z[a]*model.z[b] for a in model.ativos for b in model.ativos)  <= 1
    model.r_res_z_retorno = pyo.Constraint(rule=restricao_z_retorno)

    ## OBJETIVOS:  1 para pegar z min, 2 para z max, 3 para normalização
    def obj_1(model):
            var = sum(model.z[a] * model.retornos_ativos[d,a] for a in model.ativos for d in model.dias)
            score = sum(model.z[a] * model.score[a] for a in model.ativos)
            return model.theta*var + (1-model.theta)*score
    model.obj1 = pyo.Objective(rule=obj_1, sense=pyo.maximize)


    # NOTEBOOOK
    opt = SolverFactory('cplex', executable='C:\\CPLEX_Studio2211\\cplex\\bin\\x64_win64\\cplex.exe')

    # ------------------- solver 1
    # opt = SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')
    res = opt.solve(model,tee=False)
#     model.write('modelo_debug.lp', io_options={'symbolic_solver_labels': True})
   
    melhor_pesos = {list(model.nome_ativos.data())[a]:(model.z[a].value/model.k.value) for a in model.ativos}
    carteiras_anuais[int(ano_int)+1] = {
    	'pesos':  melhor_pesos,
    	# 'sharpe_anual': s_lo*np.sqrt(252),
    }
    print(f"{ano_int} -> {melhor_pesos}")
    print("+-="*30)
    if 'model' in locals():
    	del model
    	print('DELETADO DPS DO WHILE')
    else:
    	print("Nao consta model")
    	pass



=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
Anos totais:  ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
COMEÇANDO ANO NOVO:  2015
Nao consta model
# ------ CRIAÇÃO DO MODELO
## Carteira criada para o ano:  2016
## UTILIZANDO DADOS DE RETORNO DE:  2015
2015 -> {'ABEV3': 0.0, 'ALOS3': 0.017510503138163613, 'ALPA4': 0.08318786642802377, 'ANIM3': 0.0, 'AXIA3': 0.0, 'AZZA3': 0.0, 'B3SA3': 0.0, 'BEEF3': 0.0, 'CMIG4': 0.0, 'COGN3': 0.0, 'CPFE3': 0.0, 'CPLE3': 0.0, 'CSAN3': 0.01773276694988853, 'CSMG3': 0.0, 'CSNA3': 0.0, 'CYRE3': 0.03758811369526915, 'DIRR3': 0.0, 'ECOR3': 0.0, 'EGIE3': 0.0, 'EMBJ3': 0.20000000103905596, 'ENEV3': 0.036196854290350286, 'EQTL3': 0.0, 'EZTC3': 0.0, 'FLRY3': 0.01864009068772122, 'GGBR4': 0.0, 'GOAU4': 0.06890488591949104, 'HYPE3': 0.0, 'ISAE4': 0.0, 'ITSA4': 0.0, 'JHSF3': 0

In [7]:
carteiras_anuais

{2016: {'pesos': {'ABEV3': 0.0,
   'ALOS3': 0.017510503138163613,
   'ALPA4': 0.08318786642802377,
   'ANIM3': 0.0,
   'AXIA3': 0.0,
   'AZZA3': 0.0,
   'B3SA3': 0.0,
   'BEEF3': 0.0,
   'CMIG4': 0.0,
   'COGN3': 0.0,
   'CPFE3': 0.0,
   'CPLE3': 0.0,
   'CSAN3': 0.01773276694988853,
   'CSMG3': 0.0,
   'CSNA3': 0.0,
   'CYRE3': 0.03758811369526915,
   'DIRR3': 0.0,
   'ECOR3': 0.0,
   'EGIE3': 0.0,
   'EMBJ3': 0.20000000103905596,
   'ENEV3': 0.036196854290350286,
   'EQTL3': 0.0,
   'EZTC3': 0.0,
   'FLRY3': 0.01864009068772122,
   'GGBR4': 0.0,
   'GOAU4': 0.06890488591949104,
   'HYPE3': 0.0,
   'ISAE4': 0.0,
   'ITSA4': 0.0,
   'JHSF3': 0.03906520844874571,
   'KLBN11': 0.0,
   'LREN3': 0.0,
   'MBRF3': 0.0,
   'MGLU3': 0.1079357620727159,
   'MOTV3': 0.0,
   'MRVE3': 0.02884982613780605,
   'MULT3': 0.0,
   'NATU3': 0.0,
   'PETR3': 0.0,
   'PETR4': 0.0,
   'POMO4': 0.09123420919376098,
   'PRIO3': 0.0,
   'RADL3': 0.0,
   'RAIL3': 0.08509690214930055,
   'RENT3': 0.0,
   'SBSP3'

In [8]:
linhas = []
for an in anos:
    an = int(an.split('-')[0])+1
    print(an)
    for k, v in carteiras_anuais[an]['pesos'].items():
        # if v >= vb_peso_minimo and v <= vb_peso_maximo:
        if round(v, 4)>0.000 and v <= vb_peso_maximo*1.10:
            linhas.append({'ano': an, 'ativo': k, 'peso': v})

df_portfolios = pd.DataFrame(linhas)

# # visões instantâneas:
# df_portfolios[df_portfolios['ano'] == 2016].sort_values('peso', ascending=False)  # uma carteira
# df_portfolios.pivot(index='ativo', columns='ano', values='peso')                  # matriz ativo × ano
# df_portfolios.groupby('ativo')['ano'].count().sort_values(ascending=False)  


2016
2017
2018
2019
2020
2021
2022
2023
2024
2025
2026


In [11]:
df_portfolios[df_portfolios['ano'] == 2025].sort_values('peso', ascending=False)['peso']

119    0.130660
115    0.122947
114    0.117447
123    0.111532
111    0.088797
121    0.087647
116    0.079952
118    0.065770
113    0.061791
122    0.051906
112    0.050696
117    0.019437
120    0.011417
Name: peso, dtype: float64

In [10]:
df_portfolios.to_csv('../carteiras/linear_sharpe_piotroski/carteiras_sharpe_linear_piotroski.csv')
